In [1]:
import sys

sys.path.append("..")

from common.metrics import metrics
from common.preprocessing import preprocessing_rg3
from common.train_test_split import split_data
from common.model_composition import model_composition

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
rg3 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_rg3_다시되돌림.csv")

| 제품 | 시스템 처리 | 표시 내용 |
|---|---|---|
| RH | 공정값으로 위험도 산출·검사 우선순위 결정 | 위험 점수와 우선검사 여부 |
| LH | 자동 정상·불량 판정 보류 | **“LH 불량 학습 사례가 없어 AI 판정 대상에서 제외됩니다. 기존 품질검사를 진행하세요.”** |

# RG3 모델링 방향 재정립

Timestamp는 불량 예측 변수로 사용하지 않는다.

Timestamp의 역할은 생산 순서를 정렬하여
Lag, Rolling, EWMA 등 이전 Cycle 기반 파생변수를
생성하기 위한 보조 정보로 제한한다.

RG3 데이터는 전체 불량 데이터 수가 매우 적기 때문에
생산 Session을 기준으로 데이터를 강하게 분리할 경우
각 Train / Validation 구간에 포함되는 불량 수가 지나치게 감소하여
평가 결과의 변동성이 크게 증가한다.

따라서 Feature 및 모델 선택의 주 평가 방법은
Repeated Stratified Cross Validation으로 설정한다.

생산 Session 기반 검증은 최종 모델의
시간적 강건성을 확인하기 위한 추가 분석으로 활용한다.

현재까지의 실험에서는
최근 약 5 Cycle의 공정 상태 대비 현재 센서값의
절대 이탈 정도를 나타내는 Absolute EWMA5 Feature가
가장 높은 성능을 보였다.

따라서 이후 분석은 Timestamp 자체가 아닌
공정 변수와 공정 변수 간 관계,
그리고 최근 공정 상태 대비 변화량을 중심으로 진행한다.

# RG3 최종 전처리 및 Feature Engineering

RG3 데이터의 Timestamp는 불량을 직접 예측하는 Feature로 사용하지 않는다.

Timestamp는 생산 순서를 정렬하고,
직전 생산 Cycle을 기준으로 EWMA 등의 동적 Feature를 생성하기 위한
보조 정보로만 활용한다.

또한 장시간 생산이 중단된 이후 이전 Cycle의 상태가
현재 생산 상태에 영향을 주지 않도록
30분 이상 생산 간격이 발생하면 새로운 Run으로 구분한다.

단, Run 정보 역시 모델 Feature로 사용하지 않는다.

RG3 원본 공정 변수는 이미 Standard Scaling되어 있으므로
추가 Standard Scaling은 수행하지 않는다.

Feature Engineering은 다음 네 가지 축으로 구성한다.

1. Original Process Feature
2. 최근 5 Cycle 대비 Absolute EWMA Deviation
3. 공정 변수 간 관계를 표현하는 Interaction Feature
4. 공정 영역별 Instability Feature

Standard Scaling된 변수 간 단순 비율은 물리적 의미를 해석하기 어렵기 때문에
Ratio Feature는 생성하지 않고,
차이·곱·통계적 분산 구조를 중심으로 Feature를 생성한다.

In [3]:
import numpy as np
import pandas as pd


# ============================================================
# 0. 원본 데이터 복사
# ============================================================

rg3_prep = rg3.copy()

print("Original Shape:", rg3_prep.shape)


# ============================================================
# 1. Timestamp 처리
# ============================================================

rg3_prep["TimeStamp"] = pd.to_datetime(
    rg3_prep["TimeStamp"],
    errors="coerce"
)

rg3_prep = (
    rg3_prep
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


# ============================================================
# 2. RG3 RH 데이터만 사용
# ============================================================

rg3_rh = (
    rg3_prep[
        rg3_prep["RH_LH"] == "RH"
    ]
    .copy()
    .reset_index(drop=True)
)

print("RH Shape:", rg3_rh.shape)

print("\nTarget:")
print(
    rg3_rh["PassOrFail"]
    .value_counts()
)

print("\nTarget Ratio:")
print(
    rg3_rh["PassOrFail"]
    .value_counts(normalize=True)
)


# ============================================================
# 3. 원본 공정 변수 정의
# ============================================================

PROCESS_COLS_ALL = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Clamp_Open_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4"
]


# ============================================================
# 4. 상수 컬럼 자동 제거
# ============================================================

CONSTANT_COLS = [
    col
    for col in PROCESS_COLS_ALL
    if rg3_rh[col].nunique(dropna=False) <= 1
]

PROCESS_COLS = [
    col
    for col in PROCESS_COLS_ALL
    if col not in CONSTANT_COLS
]

print("\nConstant Columns:")
print(CONSTANT_COLS)

print("\nProcess Feature Count:")
print(len(PROCESS_COLS))


# ============================================================
# 5. 결측 / Infinite 확인
# ============================================================

print(
    "\nMissing:",
    rg3_rh[PROCESS_COLS]
    .isna()
    .sum()
    .sum()
)

print(
    "Infinite:",
    np.isinf(
        rg3_rh[PROCESS_COLS]
        .to_numpy(dtype=float)
    ).sum()
)


# ============================================================
# 6. 생산 순서용 Run 생성
#
# 모델 Feature로 사용하지 않음.
# 단지 장시간 생산 중단 이후 EWMA를 리셋하기 위한 용도.
# ============================================================

rg3_rh["Time_Diff"] = (
    rg3_rh["TimeStamp"]
    .diff()
)

NEW_RUN = (
    rg3_rh["Time_Diff"].isna()
    |
    (
        rg3_rh["Time_Diff"]
        >= pd.Timedelta(minutes=30)
    )
)

rg3_rh["Run_ID"] = (
    NEW_RUN
    .cumsum()
    .astype(int)
)

print("\nRun Count:")
print(rg3_rh["Run_ID"].nunique())


# ============================================================
# 7. 공정 관계 Feature
#
# 주의:
# 원본값이 Standard Scaling되어 있기 때문에
# 아래 Feature는 실제 물리 단위의 차이/비율이 아니라
# 표준화된 공정 상태 간 관계를 표현하는 통계적 Feature이다.
# ============================================================


# ------------------------------------------------------------
# 7-1. RPM / Plasticizing
# ------------------------------------------------------------

rg3_rh["RPM_Z_Contrast"] = (
    rg3_rh["Max_Screw_RPM"]
    - rg3_rh["Average_Screw_RPM"]
)

rg3_rh["RPM_Z_Abs_Contrast"] = (
    rg3_rh["RPM_Z_Contrast"]
    .abs()
)

rg3_rh["RPM_Z_Interaction"] = (
    rg3_rh["Max_Screw_RPM"]
    * rg3_rh["Average_Screw_RPM"]
)

rg3_rh["PlasticizingTime_x_AvgRPM"] = (
    rg3_rh["Plasticizing_Time"]
    * rg3_rh["Average_Screw_RPM"]
)

rg3_rh["PlasticizingTime_x_MaxRPM"] = (
    rg3_rh["Plasticizing_Time"]
    * rg3_rh["Max_Screw_RPM"]
)


# ------------------------------------------------------------
# 7-2. Pressure / Injection Speed
# ------------------------------------------------------------

rg3_rh["Injection_SwitchPressure_Z_Contrast"] = (
    rg3_rh["Max_Injection_Pressure"]
    - rg3_rh["Max_Switch_Over_Pressure"]
)

rg3_rh["BackPressure_Z_Contrast"] = (
    rg3_rh["Max_Back_Pressure"]
    - rg3_rh["Average_Back_Pressure"]
)

rg3_rh["SwitchPressure_x_InjectionSpeed"] = (
    rg3_rh["Max_Switch_Over_Pressure"]
    * rg3_rh["Max_Injection_Speed"]
)

rg3_rh["InjectionPressure_x_InjectionSpeed"] = (
    rg3_rh["Max_Injection_Pressure"]
    * rg3_rh["Max_Injection_Speed"]
)


# ------------------------------------------------------------
# 7-3. Position
# ------------------------------------------------------------

rg3_rh["Position_Z_Contrast"] = (
    rg3_rh["Plasticizing_Position"]
    - rg3_rh["Cushion_Position"]
)

rg3_rh["Position_Z_Abs_Contrast"] = (
    rg3_rh["Position_Z_Contrast"]
    .abs()
)

rg3_rh["PlasticizingPos_x_SwitchPressure"] = (
    rg3_rh["Plasticizing_Position"]
    * rg3_rh["Max_Switch_Over_Pressure"]
)

rg3_rh["PlasticizingPos_x_InjectionSpeed"] = (
    rg3_rh["Plasticizing_Position"]
    * rg3_rh["Max_Injection_Speed"]
)


# ------------------------------------------------------------
# 7-4. Barrel Temperature
# ------------------------------------------------------------

BARREL_COLS = [
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6"
]

rg3_rh["Barrel_Z_Mean"] = (
    rg3_rh[BARREL_COLS]
    .mean(axis=1)
)

rg3_rh["Barrel_Z_Std"] = (
    rg3_rh[BARREL_COLS]
    .std(axis=1)
)

rg3_rh["Barrel_Z_Range"] = (
    rg3_rh[BARREL_COLS]
    .max(axis=1)
    - rg3_rh[BARREL_COLS]
    .min(axis=1)
)

rg3_rh["Barrel_FrontRear_Z_Contrast"] = (
    rg3_rh[
        [
            "Barrel_Temperature_1",
            "Barrel_Temperature_2"
        ]
    ].mean(axis=1)
    -
    rg3_rh[
        [
            "Barrel_Temperature_5",
            "Barrel_Temperature_6"
        ]
    ].mean(axis=1)
)


# ------------------------------------------------------------
# 7-5. Mold Temperature
# ------------------------------------------------------------

rg3_rh["Mold_Z_Mean"] = (
    rg3_rh[
        [
            "Mold_Temperature_3",
            "Mold_Temperature_4"
        ]
    ]
    .mean(axis=1)
)

rg3_rh["Mold_Z_Contrast"] = (
    rg3_rh["Mold_Temperature_3"]
    - rg3_rh["Mold_Temperature_4"]
)

rg3_rh["Mold_Z_Abs_Contrast"] = (
    rg3_rh["Mold_Z_Contrast"]
    .abs()
)


# ------------------------------------------------------------
# 7-6. Time 변수 Interaction
#
# 이미 각각 Standard Scaling되었으므로
# 시간끼리 직접 빼서 "실제 초 차이"로 해석하지 않는다.
# 대신 동시에 높은/낮은 상태를 표현하는 Interaction 사용.
# ------------------------------------------------------------

rg3_rh["InjectionTime_x_FillingTime"] = (
    rg3_rh["Injection_Time"]
    * rg3_rh["Filling_Time"]
)

rg3_rh["PlasticizingTime_x_CycleTime"] = (
    rg3_rh["Plasticizing_Time"]
    * rg3_rh["Cycle_Time"]
)


# ============================================================
# 8. Relation Feature 목록
# ============================================================

RELATION_COLS = [
    "RPM_Z_Contrast",
    "RPM_Z_Abs_Contrast",
    "RPM_Z_Interaction",
    "PlasticizingTime_x_AvgRPM",
    "PlasticizingTime_x_MaxRPM",

    "Injection_SwitchPressure_Z_Contrast",
    "BackPressure_Z_Contrast",
    "SwitchPressure_x_InjectionSpeed",
    "InjectionPressure_x_InjectionSpeed",

    "Position_Z_Contrast",
    "Position_Z_Abs_Contrast",
    "PlasticizingPos_x_SwitchPressure",
    "PlasticizingPos_x_InjectionSpeed",

    "Barrel_Z_Mean",
    "Barrel_Z_Std",
    "Barrel_Z_Range",
    "Barrel_FrontRear_Z_Contrast",

    "Mold_Z_Mean",
    "Mold_Z_Contrast",
    "Mold_Z_Abs_Contrast",

    "InjectionTime_x_FillingTime",
    "PlasticizingTime_x_CycleTime"
]

print(
    "\nRelation Feature Count:",
    len(RELATION_COLS)
)


# ============================================================
# 9. Previous-only EWMA5
#
# 현재 Cycle은 기준 계산에서 제외.
#
# current - EWMA(previous cycles)
#
# 이후 abs() 적용.
# ============================================================

EWMA_SPAN = 5


def create_abs_ewma_features(
    df,
    columns,
    span=5,
    group_col="Run_ID"
):

    new_cols = []

    for col in columns:

        baseline_col = (
            f"{col}_ewma{span}_baseline"
        )

        abs_dev_col = (
            f"{col}_ewma{span}_abs_dev"
        )

        df[baseline_col] = (
            df.groupby(
                group_col,
                sort=False
            )[col]
            .transform(
                lambda x:
                    x.shift(1)
                    .ewm(
                        span=span,
                        adjust=False
                    )
                    .mean()
            )
        )

        df[abs_dev_col] = (
            df[col]
            - df[baseline_col]
        ).abs()

        new_cols.append(
            abs_dev_col
        )

    return new_cols


# ============================================================
# 10. Original Sensor Abs EWMA5
# ============================================================

ABS_EWMA5_COLS = (
    create_abs_ewma_features(
        rg3_rh,
        PROCESS_COLS,
        span=5
    )
)

print(
    "\nOriginal Abs EWMA5:",
    len(ABS_EWMA5_COLS)
)


# ============================================================
# 11. Relation Feature 자체의 Abs EWMA5
# ============================================================

RELATION_ABS_EWMA5_COLS = (
    create_abs_ewma_features(
        rg3_rh,
        RELATION_COLS,
        span=5
    )
)

print(
    "Relation Abs EWMA5:",
    len(RELATION_ABS_EWMA5_COLS)
)


# ============================================================
# 12. Run 첫 Cycle의 EWMA NaN 처리
#
# 이전 생산 이력이 없으므로 0으로 설정.
# 모델에 Timestamp / Run_ID 자체는 넣지 않는다.
# ============================================================

ALL_EWMA_COLS = (
    ABS_EWMA5_COLS
    + RELATION_ABS_EWMA5_COLS
)

rg3_rh[ALL_EWMA_COLS] = (
    rg3_rh[ALL_EWMA_COLS]
    .fillna(0)
)


# ============================================================
# 13. 공정 영역별 Instability
#
# 개별 센서의 Abs EWMA5를 묶어
# 공정 영역 전체의 흔들림 정도를 표현.
# ============================================================


TIME_BASE = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time"
]

POSITION_BASE = [
    "Cushion_Position",
    "Plasticizing_Position"
]

RPM_BASE = [
    "Max_Screw_RPM",
    "Average_Screw_RPM"
]

PRESSURE_BASE = [
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure"
]

BARREL_BASE = BARREL_COLS

MOLD_BASE = [
    "Mold_Temperature_3",
    "Mold_Temperature_4"
]


def ewma_names(cols):

    return [
        f"{col}_ewma5_abs_dev"
        for col in cols
        if f"{col}_ewma5_abs_dev"
        in rg3_rh.columns
    ]


rg3_rh["Time_Instability"] = (
    rg3_rh[
        ewma_names(TIME_BASE)
    ]
    .mean(axis=1)
)

rg3_rh["Position_Instability"] = (
    rg3_rh[
        ewma_names(POSITION_BASE)
    ]
    .mean(axis=1)
)

rg3_rh["RPM_Instability"] = (
    rg3_rh[
        ewma_names(RPM_BASE)
    ]
    .mean(axis=1)
)

rg3_rh["Pressure_Instability"] = (
    rg3_rh[
        ewma_names(PRESSURE_BASE)
    ]
    .mean(axis=1)
)

rg3_rh["BarrelTemp_Instability"] = (
    rg3_rh[
        ewma_names(BARREL_BASE)
    ]
    .mean(axis=1)
)

rg3_rh["MoldTemp_Instability"] = (
    rg3_rh[
        ewma_names(MOLD_BASE)
    ]
    .mean(axis=1)
)


# ============================================================
# 14. 전체 공정 Instability
# ============================================================

rg3_rh["Global_Instability_Mean"] = (
    rg3_rh[ABS_EWMA5_COLS]
    .mean(axis=1)
)

rg3_rh["Global_Instability_Max"] = (
    rg3_rh[ABS_EWMA5_COLS]
    .max(axis=1)
)

rg3_rh["Global_Instability_Std"] = (
    rg3_rh[ABS_EWMA5_COLS]
    .std(axis=1)
)


INSTABILITY_COLS = [
    "Time_Instability",
    "Position_Instability",
    "RPM_Instability",
    "Pressure_Instability",
    "BarrelTemp_Instability",
    "MoldTemp_Instability",
    "Global_Instability_Mean",
    "Global_Instability_Max",
    "Global_Instability_Std"
]


# ============================================================
# 15. Feature Set 구성
#
# 모든 걸 한꺼번에 넣지 않고
# 어떤 종류가 실제 성능을 개선하는지 단계적으로 비교.
# ============================================================

FEATURE_SETS = {

    # --------------------------------------------------------
    # Baseline
    # --------------------------------------------------------
    "Original":
        PROCESS_COLS,


    # --------------------------------------------------------
    # 현재까지 가장 강했던 Dynamic Feature
    # --------------------------------------------------------
    "Abs EWMA5":
        ABS_EWMA5_COLS,


    # --------------------------------------------------------
    # 기존 최고 조합
    # --------------------------------------------------------
    "Original + Abs EWMA5":
        PROCESS_COLS
        + ABS_EWMA5_COLS,


    # --------------------------------------------------------
    # 공정 변수 관계만
    # --------------------------------------------------------
    "Relations":
        RELATION_COLS,


    # --------------------------------------------------------
    # Original + 공정 관계
    # --------------------------------------------------------
    "Original + Relations":
        PROCESS_COLS
        + RELATION_COLS,


    # --------------------------------------------------------
    # EWMA + 공정 관계
    # --------------------------------------------------------
    "Abs EWMA5 + Relations":
        ABS_EWMA5_COLS
        + RELATION_COLS,


    # --------------------------------------------------------
    # 공정 관계 자체의 변화까지 포함
    # --------------------------------------------------------
    "Abs EWMA5 + Relation EWMA5":
        ABS_EWMA5_COLS
        + RELATION_ABS_EWMA5_COLS,


    # --------------------------------------------------------
    # 공정 영역별 흔들림
    # --------------------------------------------------------
    "Abs EWMA5 + Instability":
        ABS_EWMA5_COLS
        + INSTABILITY_COLS,


    # --------------------------------------------------------
    # 종합 후보
    # --------------------------------------------------------
    "Original + Abs EWMA5 + Relations":
        PROCESS_COLS
        + ABS_EWMA5_COLS
        + RELATION_COLS
}


# ============================================================
# 16. 중복 Feature 제거
# ============================================================

for name in FEATURE_SETS:

    FEATURE_SETS[name] = list(
        dict.fromkeys(
            FEATURE_SETS[name]
        )
    )


# ============================================================
# 17. 최종 결측 / Infinite 처리
# ============================================================

ALL_MODEL_FEATURES = list(
    dict.fromkeys(
        col
        for cols in FEATURE_SETS.values()
        for col in cols
    )
)

rg3_rh[
    ALL_MODEL_FEATURES
] = (
    rg3_rh[
        ALL_MODEL_FEATURES
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)


# ============================================================
# 18. 전처리 결과 확인
# ============================================================

print("\n========================================")
print("RG3 PREPROCESSING COMPLETE")
print("========================================")

print(
    "Rows:",
    len(rg3_rh)
)

print(
    "Normal:",
    (rg3_rh["PassOrFail"] == 0).sum()
)

print(
    "Defect:",
    (rg3_rh["PassOrFail"] == 1).sum()
)

print(
    "Original Features:",
    len(PROCESS_COLS)
)

print(
    "Relation Features:",
    len(RELATION_COLS)
)

print(
    "Original Abs EWMA5:",
    len(ABS_EWMA5_COLS)
)

print(
    "Relation Abs EWMA5:",
    len(RELATION_ABS_EWMA5_COLS)
)

print(
    "Instability Features:",
    len(INSTABILITY_COLS)
)

print(
    "Total Candidate Features:",
    len(ALL_MODEL_FEATURES)
)

print(
    "Missing:",
    rg3_rh[
        ALL_MODEL_FEATURES
    ]
    .isna()
    .sum()
    .sum()
)


print("\n========================================")
print("FEATURE SETS")
print("========================================")

for name, cols in FEATURE_SETS.items():

    print(
        f"{name:<40}",
        len(cols)
    )

Original Shape: (1182, 29)
RH Shape: (591, 29)

Target:
PassOrFail
0    566
1     25
Name: count, dtype: int64

Target Ratio:
PassOrFail
0    0.957699
1    0.042301
Name: proportion, dtype: float64

Constant Columns:
['Clamp_Open_Position']

Process Feature Count:
23

Missing: 0
Infinite: 0

Run Count:
5

Relation Feature Count: 22

Original Abs EWMA5: 23
Relation Abs EWMA5: 22

RG3 PREPROCESSING COMPLETE
Rows: 591
Normal: 566
Defect: 25
Original Features: 23
Relation Features: 22
Original Abs EWMA5: 23
Relation Abs EWMA5: 22
Instability Features: 9
Total Candidate Features: 99
Missing: 0

FEATURE SETS
Original                                 23
Abs EWMA5                                23
Original + Abs EWMA5                     46
Relations                                22
Original + Relations                     45
Abs EWMA5 + Relations                    45
Abs EWMA5 + Relation EWMA5               45
Abs EWMA5 + Instability                  32
Original + Abs EWMA5 + Relations      

C:\Users\Administrator\AppData\Local\Temp\ipykernel_33776\210997266.py:433: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[abs_dev_col] = (
C:\Users\Administrator\AppData\Local\Temp\ipykernel_33776\210997266.py:417: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[baseline_col] = (
C:\Users\Administrator\AppData\Local\Temp\ipykernel_33776\210997266.py:550: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all colum

# 20. RG3 Feature Set 성능 비교

전처리 단계에서 생성한 Feature Set들이
실제로 불량 분류 성능을 향상시키는지 비교한다.

RG3 데이터는 정상 566건, 불량 25건으로
불량 비율이 약 4.23%인 불균형 데이터이다.

따라서 Accuracy보다 불량 클래스의 탐지 성능을
더 적절하게 평가할 수 있는 PR-AUC를 주요 지표로 사용한다.

### 평가 방법

- Model : XGBoost
- Validation : Repeated Stratified K-Fold
- n_splits : 3
- n_repeats : 10
- 총 평가 횟수 : 30 Fold
- 주요 지표 : PR-AUC
- 보조 지표 : ROC-AUC
- Class Imbalance : scale_pos_weight 적용

### 비교 Feature Set

1. Original
2. Abs EWMA5
3. Original + Abs EWMA5
4. Relations
5. Original + Relations
6. Abs EWMA5 + Relations
7. Abs EWMA5 + Relation EWMA5
8. Abs EWMA5 + Instability
9. Original + Abs EWMA5 + Relations

모든 Feature Set은 동일한 Fold를 사용하여
Feature 구성만 달라졌을 때의 성능 차이를 비교한다.

In [4]:
rg3_rh = rg3_rh.copy()

print("Shape:", rg3_rh.shape)
print("Defect Rate:", rg3_rh["PassOrFail"].mean())

Shape: (591, 152)
Defect Rate: 0.04230118443316413


In [5]:
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

from xgboost import XGBClassifier

In [6]:
y_all = (
    rg3_rh["PassOrFail"]
    .reset_index(drop=True)
)

rskf = RepeatedStratifiedKFold(
    n_splits=3,
    n_repeats=10,
    random_state=42
)

cv_splits = list(
    rskf.split(
        np.zeros(len(y_all)),
        y_all
    )
)

print("Total CV Folds:", len(cv_splits))

Total CV Folds: 30


In [7]:
feature_set_results = []

for feature_name, feature_cols in FEATURE_SETS.items():

    print(
        f"Running: {feature_name} "
        f"({len(feature_cols)} features)"
    )

    X_temp = (
        rg3_rh[feature_cols]
        .reset_index(drop=True)
    )

    y_temp = (
        rg3_rh["PassOrFail"]
        .reset_index(drop=True)
    )

    for fold_num, (train_idx, valid_idx) in enumerate(
        cv_splits,
        start=1
    ):

        X_train = X_temp.iloc[train_idx]
        X_valid = X_temp.iloc[valid_idx]

        y_train = y_temp.iloc[train_idx]
        y_valid = y_temp.iloc[valid_idx]

        neg = (y_train == 0).sum()
        pos = (y_train == 1).sum()

        model = XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.03,

            subsample=0.8,
            colsample_bytree=0.8,

            scale_pos_weight=neg / pos,

            eval_metric="logloss",

            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train
        )

        valid_score = model.predict_proba(
            X_valid
        )[:, 1]

        pr_auc = average_precision_score(
            y_valid,
            valid_score
        )

        roc_auc = roc_auc_score(
            y_valid,
            valid_score
        )

        feature_set_results.append({
            "Feature_Set": feature_name,
            "Fold": fold_num,

            "PR_AUC": pr_auc,
            "ROC_AUC": roc_auc
        })

Running: Original (23 features)
Running: Abs EWMA5 (23 features)
Running: Original + Abs EWMA5 (46 features)
Running: Relations (22 features)
Running: Original + Relations (45 features)
Running: Abs EWMA5 + Relations (45 features)
Running: Abs EWMA5 + Relation EWMA5 (45 features)
Running: Abs EWMA5 + Instability (32 features)
Running: Original + Abs EWMA5 + Relations (68 features)


In [8]:
feature_set_results = pd.DataFrame(
    feature_set_results
)

feature_set_summary = (
    feature_set_results
    .groupby("Feature_Set")
    .agg(
        PR_AUC_Mean=("PR_AUC", "mean"),
        PR_AUC_Std=("PR_AUC", "std"),
        PR_AUC_Min=("PR_AUC", "min"),
        PR_AUC_Max=("PR_AUC", "max"),

        ROC_AUC_Mean=("ROC_AUC", "mean"),
        ROC_AUC_Std=("ROC_AUC", "std")
    )
    .sort_values(
        "PR_AUC_Mean",
        ascending=False
    )
)

feature_set_summary

,PR_AUC_Mean,PR_AUC_Std,PR_AUC_Min,PR_AUC_Max,ROC_AUC_Mean,ROC_AUC_Std
Feature_Set,,,,,,
Original,0.080049,0.049817,0.031940,0.234006,0.521028,0.073881
Original + Abs EWMA5,0.079661,0.044378,0.040400,0.265566,0.544369,0.088600
Abs EWMA5 + Relation EWMA5,0.076763,0.035022,0.037926,0.178457,0.560066,0.092302
Abs EWMA5,0.071225,0.023871,0.037383,0.124670,0.573832,0.076470
Abs EWMA5 + Relations,0.065533,0.019449,0.043524,0.135668,0.567425,0.070980
Abs EWMA5 + Instability,0.065529,0.016556,0.042170,0.102409,0.575760,0.082334
Original + Abs EWMA5 + Relations,0.064902,0.019392,0.041684,0.140502,0.552105,0.076645
Original + Relations,0.053577,0.022590,0.032075,0.127344,0.482834,0.082809
Relations,0.043524,0.009874,0.029773,0.066753,0.446028,0.076393


In [9]:
random_pr_baseline = (
    rg3_rh["PassOrFail"].mean()
)

feature_set_summary[
    "PR_Lift"
] = (
    feature_set_summary["PR_AUC_Mean"]
    / random_pr_baseline
)

feature_set_summary = (
    feature_set_summary
    .sort_values(
        "PR_AUC_Mean",
        ascending=False
    )
)

feature_set_summary

,PR_AUC_Mean,PR_AUC_Std,PR_AUC_Min,PR_AUC_Max,ROC_AUC_Mean,ROC_AUC_Std,PR_Lift
Feature_Set,,,,,,,
Original,0.080049,0.049817,0.031940,0.234006,0.521028,0.073881,1.892366
Original + Abs EWMA5,0.079661,0.044378,0.040400,0.265566,0.544369,0.088600,1.883175
Abs EWMA5 + Relation EWMA5,0.076763,0.035022,0.037926,0.178457,0.560066,0.092302,1.814667
Abs EWMA5,0.071225,0.023871,0.037383,0.124670,0.573832,0.076470,1.683763
Abs EWMA5 + Relations,0.065533,0.019449,0.043524,0.135668,0.567425,0.070980,1.549189
Abs EWMA5 + Instability,0.065529,0.016556,0.042170,0.102409,0.575760,0.082334,1.549111
Original + Abs EWMA5 + Relations,0.064902,0.019392,0.041684,0.140502,0.552105,0.076645,1.534274
Original + Relations,0.053577,0.022590,0.032075,0.127344,0.482834,0.082809,1.266557
Relations,0.043524,0.009874,0.029773,0.066753,0.446028,0.076393,1.028904


In [10]:
original_scores = (
    feature_set_results[
        feature_set_results["Feature_Set"] == "Original"
    ][
        ["Fold", "PR_AUC"]
    ]
    .rename(
        columns={
            "PR_AUC": "Original_PR_AUC"
        }
    )
)

In [11]:
win_rows = []

for feature_name in FEATURE_SETS.keys():

    if feature_name == "Original":
        continue

    temp = (
        feature_set_results[
            feature_set_results["Feature_Set"] == feature_name
        ][
            ["Fold", "PR_AUC"]
        ]
        .rename(
            columns={
                "PR_AUC": "Feature_PR_AUC"
            }
        )
    )

    compare = original_scores.merge(
        temp,
        on="Fold"
    )

    compare["Delta"] = (
        compare["Feature_PR_AUC"]
        - compare["Original_PR_AUC"]
    )

    win_rows.append({
        "Feature_Set": feature_name,

        "Win": int(
            (compare["Delta"] > 0).sum()
        ),

        "Tie": int(
            (compare["Delta"] == 0).sum()
        ),

        "Lose": int(
            (compare["Delta"] < 0).sum()
        ),

        "Mean_Delta":
            compare["Delta"].mean(),

        "Median_Delta":
            compare["Delta"].median()
    })

In [12]:
feature_win_summary = (
    pd.DataFrame(win_rows)
    .sort_values(
        "Mean_Delta",
        ascending=False
    )
)

feature_win_summary

,Feature_Set,Win,Tie,Lose,Mean_Delta,Median_Delta
1,Original + Abs EWMA5,17,0,13,-0.000389,0.006049
5,Abs EWMA5 + Relation EWMA5,15,0,15,-0.003287,-0.000066
0,Abs EWMA5,16,0,14,-0.008824,0.005038
4,Abs EWMA5 + Relations,16,0,14,-0.014517,0.000716
6,Abs EWMA5 + Instability,14,0,16,-0.014520,-0.001262
7,Original + Abs EWMA5 + Relations,16,0,14,-0.015148,0.001948
3,Original + Relations,2,0,28,-0.026472,-0.012261
2,Relations,1,0,29,-0.036525,-0.020386


## 20-7. Feature Set 비교 결과

전체 RG3 RH 데이터 591건을 대상으로
Repeated Stratified Cross Validation을 수행하였다.

### 주요 결과

| Feature Set | PR-AUC Mean | PR Lift |
|---|---:|---:|
| Original | 0.0800 | 1.89 |
| Original + Abs EWMA5 | 0.0797 | 1.88 |
| Abs EWMA5 + Relation EWMA5 | 0.0768 | 1.81 |
| Abs EWMA5 | 0.0712 | 1.68 |
| Relations | 0.0435 | 1.03 |

Original Feature가 가장 높은 평균 PR-AUC를 기록하였다.

Original + Abs EWMA5는 30개 Fold 중 17개 Fold에서
Original보다 높은 성능을 보였지만,
평균 성능은 Original과 거의 동일하였다.

따라서 Absolute EWMA5는 일부 데이터 분할에서
추가적인 정보를 제공하지만,
전체 데이터에 대해 일관된 성능 향상을 제공한다고 판단하기 어렵다.

공정 변수 간 Difference, Interaction 및
공정 영역별 Instability Feature 역시
Original Feature 대비 성능을 개선하지 못하였다.

특히 Relations Feature 단독의 PR-AUC는 0.0435로,
전체 불량률 0.0423과 거의 동일하였다.

따라서 이후 모델링에서는
복잡한 공정 Interaction Feature를 제거하고
Original Process Feature를 중심으로 모델을 재검토한다.

# 21. Original Feature 기반 모델 비교

전체 RG3 RH 데이터에 대한 Feature Set 비교 결과,
Original Process Feature가 가장 높은 평균 PR-AUC를 기록하였다.

따라서 불필요한 파생변수를 추가하기보다
23개의 Original Process Feature를 기준으로
분류 알고리즘에 따른 성능 차이를 비교한다.

모든 모델은 동일한 30개의
Repeated Stratified Cross Validation Fold를 사용한다.

### 비교 모델

- XGBoost
- LightGBM
- CatBoost
- Random Forest
- Extra Trees
- Logistic Regression

주요 평가 지표는 PR-AUC로 설정한다.

In [13]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)

from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

In [14]:
X_original = (
    rg3_rh[PROCESS_COLS]
    .reset_index(drop=True)
)

y_original = (
    rg3_rh["PassOrFail"]
    .reset_index(drop=True)
)

print("X:", X_original.shape)
print("Normal:", (y_original == 0).sum())
print("Defect:", (y_original == 1).sum())

X: (591, 23)
Normal: 566
Defect: 25


In [15]:
model_results = []

for fold_num, (train_idx, valid_idx) in enumerate(
    cv_splits,
    start=1
):

    X_train = X_original.iloc[train_idx]
    X_valid = X_original.iloc[valid_idx]

    y_train = y_original.iloc[train_idx]
    y_valid = y_original.iloc[valid_idx]

    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()

    pos_weight = neg / pos

    models = {

        "XGBoost": XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ),

        "LightGBM": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            num_leaves=7,
            class_weight="balanced",
            random_state=42,
            verbosity=-1
        ),

        "CatBoost": CatBoostClassifier(
            iterations=300,
            depth=4,
            learning_rate=0.03,
            class_weights=[
                1,
                pos_weight
            ],
            random_seed=42,
            verbose=False
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=500,
            max_depth=4,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),

        "Extra Trees": ExtraTreesClassifier(
            n_estimators=500,
            max_depth=4,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),

        "Logistic Regression": Pipeline([
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=3000,
                    random_state=42
                )
            )
        ])
    }

    for model_name, model in models.items():

        model.fit(
            X_train,
            y_train
        )

        score = model.predict_proba(
            X_valid
        )[:, 1]

        model_results.append({
            "Fold": fold_num,
            "Model": model_name,

            "PR_AUC": average_precision_score(
                y_valid,
                score
            ),

            "ROC_AUC": roc_auc_score(
                y_valid,
                score
            )
        })

In [16]:
model_results = pd.DataFrame(
    model_results
)

model_summary = (
    model_results
    .groupby("Model")
    .agg(
        PR_AUC_Mean=("PR_AUC", "mean"),
        PR_AUC_Std=("PR_AUC", "std"),
        PR_AUC_Min=("PR_AUC", "min"),
        PR_AUC_Max=("PR_AUC", "max"),
        ROC_AUC_Mean=("ROC_AUC", "mean"),
        ROC_AUC_Std=("ROC_AUC", "std")
    )
    .sort_values(
        "PR_AUC_Mean",
        ascending=False
    )
)

model_summary["PR_Lift"] = (
    model_summary["PR_AUC_Mean"]
    / y_original.mean()
)

model_summary

,PR_AUC_Mean,PR_AUC_Std,PR_AUC_Min,PR_AUC_Max,ROC_AUC_Mean,ROC_AUC_Std,PR_Lift
Model,,,,,,,
XGBoost,0.080049,0.049817,0.031940,0.234006,0.521028,0.073881,1.892366
LightGBM,0.073618,0.045311,0.035728,0.252691,0.523652,0.080088,1.740328
CatBoost,0.070026,0.039504,0.030602,0.216838,0.523747,0.086225,1.655407
Random Forest,0.054134,0.019665,0.028218,0.108488,0.469686,0.085208,1.279725
Extra Trees,0.050838,0.013453,0.028899,0.082226,0.464546,0.078901,1.201801
Logistic Regression,0.048597,0.013304,0.028989,0.092431,0.464533,0.073469,1.148838


## 21-5. Original Feature 기반 모델 비교 결과

전체 RG3 RH 데이터 591건을 대상으로
23개의 Original Process Feature만 사용하여
여러 분류 모델을 비교하였다.

| Model | PR-AUC Mean | ROC-AUC Mean | PR Lift |
|---|---:|---:|---:|
| XGBoost | 0.0800 | 0.5210 | 1.89 |
| LightGBM | 0.0736 | 0.5237 | 1.74 |
| CatBoost | 0.0700 | 0.5237 | 1.66 |
| Random Forest | 0.0541 | 0.4697 | 1.28 |
| Extra Trees | 0.0508 | 0.4645 | 1.20 |
| Logistic Regression | 0.0486 | 0.4645 | 1.15 |

XGBoost가 가장 높은 평균 PR-AUC를 기록하였으나,
모델 간 성능 차이는 크지 않았다.

또한 최고 성능인 XGBoost의 PR-AUC도 0.0800으로,
전체 불량률 0.0423 대비 약 1.89배 수준이었다.

따라서 현재 성능 한계는 단순한 알고리즘 선택보다
Original Process Feature에서 정상과 불량을 구분할 수 있는
신호 자체가 약하거나 복수 형태로 존재하기 때문일 가능성이 높다.

이후에는 추가 모델 변경보다
각 Original Feature에서 불량이 정상과 어떤 방식으로 분리되는지
직접 분석하여 실제 불량 신호 후보를 탐색한다.

# 22. Original Feature별 불량 분리력 분석

모델 성능 비교 결과 알고리즘 변경만으로는
큰 성능 개선이 나타나지 않았다.

따라서 23개의 Original Process Feature 각각에 대해
정상과 불량의 분포가 실제로 얼마나 다른지 분석한다.

특히 불량이 한 방향으로만 발생하지 않을 가능성을 고려하여

- 값이 높을수록 불량인 경우
- 값이 낮을수록 불량인 경우
- 정상 중심에서 멀어질수록 불량인 경우

를 각각 평가한다.

이를 통해 평균 차이만으로 발견하기 어려운
극단값 형태의 비선형 불량 패턴을 탐색한다.

In [17]:
from scipy.stats import ks_2samp
from sklearn.metrics import average_precision_score

feature_signal_rows = []

y = rg3_rh["PassOrFail"].values

for col in PROCESS_COLS:

    normal = rg3_rh.loc[
        rg3_rh["PassOrFail"] == 0,
        col
    ]

    defect = rg3_rh.loc[
        rg3_rh["PassOrFail"] == 1,
        col
    ]

    # --------------------------------------
    # KS Test
    # --------------------------------------

    ks_stat, ks_pvalue = ks_2samp(
        normal,
        defect
    )

    # --------------------------------------
    # 단방향 PR-AUC
    #
    # High:
    # 값이 클수록 불량이라고 가정
    #
    # Low:
    # 값이 작을수록 불량이라고 가정
    # --------------------------------------

    high_score = rg3_rh[col].values
    low_score = -rg3_rh[col].values

    pr_high = average_precision_score(
        y,
        high_score
    )

    pr_low = average_precision_score(
        y,
        low_score
    )

    # --------------------------------------
    # 정상 중심에서 얼마나 멀리 떨어졌는가
    #
    # 원본 데이터가 Standard Scaling되어 있지만
    # 정상 데이터 기준으로 다시 중심/분산을 계산
    # --------------------------------------

    normal_mean = normal.mean()
    normal_std = normal.std()

    if normal_std > 0:

        abs_normal_z = np.abs(
            (
                rg3_rh[col].values
                - normal_mean
            )
            / normal_std
        )

        pr_abs = average_precision_score(
            y,
            abs_normal_z
        )

    else:

        pr_abs = np.nan

    # --------------------------------------
    # 가장 좋은 단방향
    # --------------------------------------

    if pr_high >= pr_low:
        best_direction = "High"
        best_direction_pr = pr_high
    else:
        best_direction = "Low"
        best_direction_pr = pr_low

    feature_signal_rows.append({

        "Feature": col,

        "Normal_Mean":
            normal.mean(),

        "Defect_Mean":
            defect.mean(),

        "Mean_Diff":
            defect.mean()
            - normal.mean(),

        "Abs_Mean_Diff":
            abs(
                defect.mean()
                - normal.mean()
            ),

        "Normal_Median":
            normal.median(),

        "Defect_Median":
            defect.median(),

        "KS_Stat":
            ks_stat,

        "KS_pvalue":
            ks_pvalue,

        "PR_High":
            pr_high,

        "PR_Low":
            pr_low,

        "Best_Direction":
            best_direction,

        "Best_Direction_PR":
            best_direction_pr,

        "PR_Abs_Normal_Z":
            pr_abs
    })

In [18]:
feature_signal = pd.DataFrame(
    feature_signal_rows
)

In [19]:
feature_signal[
    [
        "Feature",
        "Normal_Mean",
        "Defect_Mean",
        "Abs_Mean_Diff",
        "KS_Stat",
        "KS_pvalue"
    ]
].sort_values(
    "KS_Stat",
    ascending=False
).head(15)

,Feature,Normal_Mean,Defect_Mean,Abs_Mean_Diff,KS_Stat,KS_pvalue
18,Barrel_Temperature_5,-0.019687,0.445713,0.465400,0.188975,0.317212
22,Mold_Temperature_4,0.003555,-0.080492,0.084048,0.168905,0.450972
20,Hopper_Temperature,0.005919,-0.134011,0.139930,0.150954,0.591757
21,Mold_Temperature_3,0.003902,-0.088344,0.092246,0.145936,0.633356
7,Max_Injection_Speed,0.001844,-0.041750,0.043595,0.134205,0.730619
2,Plasticizing_Time,-0.007639,0.172956,0.180595,0.124099,0.809222
13,Average_Back_Pressure,-0.002660,0.060221,0.062881,0.121979,0.825106
16,Barrel_Temperature_3,0.006064,-0.137298,0.143362,0.119011,0.845966
19,Barrel_Temperature_6,-0.001925,0.043579,0.045504,0.112721,0.886518
17,Barrel_Temperature_4,0.002025,-0.045846,0.047871,0.106360,0.921549


In [20]:
feature_signal[
    [
        "Feature",
        "Best_Direction",
        "PR_High",
        "PR_Low",
        "Best_Direction_PR"
    ]
].sort_values(
    "Best_Direction_PR",
    ascending=False
).head(15)

,Feature,Best_Direction,PR_High,PR_Low,Best_Direction_PR
22,Mold_Temperature_4,High,0.079263,0.044576,0.079263
12,Max_Back_Pressure,High,0.073926,0.041422,0.073926
18,Barrel_Temperature_5,High,0.064997,0.031930,0.064997
17,Barrel_Temperature_4,Low,0.048880,0.054750,0.054750
14,Barrel_Temperature_1,Low,0.042920,0.054049,0.054049
13,Average_Back_Pressure,High,0.053974,0.043420,0.053974
2,Plasticizing_Time,High,0.051177,0.037312,0.051177
3,Cycle_Time,High,0.051113,0.039695,0.051113
20,Hopper_Temperature,Low,0.039045,0.050017,0.050017
16,Barrel_Temperature_3,Low,0.037609,0.048933,0.048933


In [21]:
feature_signal[
    [
        "Feature",
        "PR_Abs_Normal_Z",
        "Best_Direction_PR",
        "KS_Stat"
    ]
].sort_values(
    "PR_Abs_Normal_Z",
    ascending=False
).head(15)

,Feature,PR_Abs_Normal_Z,Best_Direction_PR,KS_Stat
17,Barrel_Temperature_4,0.057495,0.054750,0.106360
12,Max_Back_Pressure,0.053223,0.073926,0.074700
14,Barrel_Temperature_1,0.052191,0.054049,0.084028
10,Max_Injection_Pressure,0.050608,0.046152,0.067845
7,Max_Injection_Speed,0.049449,0.048256,0.134205
13,Average_Back_Pressure,0.047151,0.053974,0.121979
18,Barrel_Temperature_5,0.046473,0.064997,0.188975
22,Mold_Temperature_4,0.043437,0.079263,0.168905
8,Max_Screw_RPM,0.043111,0.045765,0.067067
20,Hopper_Temperature,0.042729,0.050017,0.150954


# 23. 공정 변수 동시 이상 개수 분석

개별 Feature의 절대값만으로 불량이 구분되지 않는 경우,
하나의 생산 Cycle에서 여러 공정 변수가 동시에
정상 범위를 벗어나는 현상이 불량 신호가 될 수 있다.

정상 제품의 분포를 기준으로 각 공정 변수의 Z-score를 계산하고,
정상 범위에서 크게 벗어난 센서의 개수를
Cycle별 이상도 Feature로 생성한다.

In [23]:
normal_data = rg3_rh[
    rg3_rh["PassOrFail"] == 0
]

normal_mean = (
    normal_data[PROCESS_COLS]
    .mean()
)

normal_std = (
    normal_data[PROCESS_COLS]
    .std()
)

In [24]:
rg3_rh = rg3_rh.copy()

print("Shape:", rg3_rh.shape)
print("Defect Rate:", rg3_rh["PassOrFail"].mean())

Shape: (591, 152)
Defect Rate: 0.04230118443316413


In [26]:
# ==========================================
# 정상 데이터 기준 Z-score 계산
# ==========================================

normal_data = rg3_rh[
    rg3_rh["PassOrFail"] == 0
].copy()

normal_mean = (
    normal_data[PROCESS_COLS]
    .mean()
)

normal_std = (
    normal_data[PROCESS_COLS]
    .std()
)

# 혹시 표준편차 0인 컬럼이 있으면 1로 대체
normal_std = normal_std.replace(
    0,
    1
)

original_normal_z = (
    rg3_rh[PROCESS_COLS]
    - normal_mean
) / normal_std

print(
    "Z-score Shape:",
    original_normal_z.shape
)

print(
    "NaN:",
    original_normal_z
    .isna()
    .sum()
    .sum()
)

Z-score Shape: (591, 23)
NaN: 0


In [27]:
rg3_rh["Original_AbsZ_Mean"] = (
    original_normal_z
    .abs()
    .mean(axis=1)
)

rg3_rh["Original_AbsZ_Max"] = (
    original_normal_z
    .abs()
    .max(axis=1)
)

rg3_rh["Original_AbsZ_Std"] = (
    original_normal_z
    .abs()
    .std(axis=1)
)

rg3_rh["Original_Outlier_Count_1_5"] = (
    original_normal_z
    .abs()
    .ge(1.5)
    .sum(axis=1)
)

rg3_rh["Original_Outlier_Count_2"] = (
    original_normal_z
    .abs()
    .ge(2.0)
    .sum(axis=1)
)

rg3_rh["Original_Outlier_Count_2_5"] = (
    original_normal_z
    .abs()
    .ge(2.5)
    .sum(axis=1)
)

In [28]:
OUTLIER_FEATURES = [
    "Original_AbsZ_Mean",
    "Original_AbsZ_Max",
    "Original_AbsZ_Std",
    "Original_Outlier_Count_1_5",
    "Original_Outlier_Count_2",
    "Original_Outlier_Count_2_5"
]

display(
    rg3_rh.groupby(
        "PassOrFail"
    )[OUTLIER_FEATURES]
    .mean()
    .T
)

PassOrFail,0,1
Original_AbsZ_Mean,0.771082,0.736972
Original_AbsZ_Max,2.180085,1.931136
Original_AbsZ_Std,0.573337,0.505961
Original_Outlier_Count_1_5,2.507067,2.240000
Original_Outlier_Count_2,0.630742,0.520000
Original_Outlier_Count_2_5,0.169611,0.080000


In [29]:
outlier_signal_rows = []

for col in OUTLIER_FEATURES:

    pr_auc = average_precision_score(
        rg3_rh["PassOrFail"],
        rg3_rh[col]
    )

    outlier_signal_rows.append({
        "Feature": col,
        "PR_AUC": pr_auc
    })

outlier_signal_result = (
    pd.DataFrame(
        outlier_signal_rows
    )
    .sort_values(
        "PR_AUC",
        ascending=False
    )
)

outlier_signal_result

,Feature,PR_AUC
5,Original_Outlier_Count_2_5,0.040778
3,Original_Outlier_Count_1_5,0.040566
0,Original_AbsZ_Mean,0.040249
4,Original_Outlier_Count_2,0.040165
1,Original_AbsZ_Max,0.037427
2,Original_AbsZ_Std,0.034969


## 23-1. 공정 변수 동시 이상 개수 분석 결과

정상 데이터의 분포를 기준으로
각 Cycle에서 공정 변수들이 정상 범위를 얼마나 벗어나는지 측정하였다.

분석한 Feature는 다음과 같다.

- 전체 공정 변수의 평균 절대 Z-score
- 최대 절대 Z-score
- 절대 Z-score의 표준편차
- |Z| ≥ 1.5인 공정 변수 개수
- |Z| ≥ 2.0인 공정 변수 개수
- |Z| ≥ 2.5인 공정 변수 개수

그러나 모든 Feature의 PR-AUC가 약 0.035~0.041 수준으로 나타났다.

전체 RG3 불량률이 약 0.0423이므로,
해당 Feature들은 Random Ranking과 유사하거나
그보다 낮은 수준의 분리력을 보였다.

따라서 RG3 불량은 여러 센서가 동시에
정상 범위를 크게 벗어나는 단순한 이상치 형태로
설명하기 어려운 것으로 판단하였다.

이후에는 전체적인 이상치 크기보다
개별 공정 변수의 특정 방향 또는 분포 형태가
불량과 관련되는지 분석한다.

In [30]:
ks_top15 = (
    feature_signal[
        [
            "Feature",
            "Normal_Mean",
            "Defect_Mean",
            "Abs_Mean_Diff",
            "KS_Stat",
            "KS_pvalue"
        ]
    ]
    .sort_values(
        "KS_Stat",
        ascending=False
    )
    .head(15)
)

ks_top15

,Feature,Normal_Mean,Defect_Mean,Abs_Mean_Diff,KS_Stat,KS_pvalue
18,Barrel_Temperature_5,-0.019687,0.445713,0.465400,0.188975,0.317212
22,Mold_Temperature_4,0.003555,-0.080492,0.084048,0.168905,0.450972
20,Hopper_Temperature,0.005919,-0.134011,0.139930,0.150954,0.591757
21,Mold_Temperature_3,0.003902,-0.088344,0.092246,0.145936,0.633356
7,Max_Injection_Speed,0.001844,-0.041750,0.043595,0.134205,0.730619
2,Plasticizing_Time,-0.007639,0.172956,0.180595,0.124099,0.809222
13,Average_Back_Pressure,-0.002660,0.060221,0.062881,0.121979,0.825106
16,Barrel_Temperature_3,0.006064,-0.137298,0.143362,0.119011,0.845966
19,Barrel_Temperature_6,-0.001925,0.043579,0.045504,0.112721,0.886518
17,Barrel_Temperature_4,0.002025,-0.045846,0.047871,0.106360,0.921549


In [31]:
direction_top15 = (
    feature_signal[
        [
            "Feature",
            "Best_Direction",
            "PR_High",
            "PR_Low",
            "Best_Direction_PR"
        ]
    ]
    .sort_values(
        "Best_Direction_PR",
        ascending=False
    )
    .head(15)
)

direction_top15

,Feature,Best_Direction,PR_High,PR_Low,Best_Direction_PR
22,Mold_Temperature_4,High,0.079263,0.044576,0.079263
12,Max_Back_Pressure,High,0.073926,0.041422,0.073926
18,Barrel_Temperature_5,High,0.064997,0.031930,0.064997
17,Barrel_Temperature_4,Low,0.048880,0.054750,0.054750
14,Barrel_Temperature_1,Low,0.042920,0.054049,0.054049
13,Average_Back_Pressure,High,0.053974,0.043420,0.053974
2,Plasticizing_Time,High,0.051177,0.037312,0.051177
3,Cycle_Time,High,0.051113,0.039695,0.051113
20,Hopper_Temperature,Low,0.039045,0.050017,0.050017
16,Barrel_Temperature_3,Low,0.037609,0.048933,0.048933


## 23-1. 공정 변수 동시 이상 개수 분석 결과

정상 데이터의 분포를 기준으로
각 Cycle에서 공정 변수들이 정상 범위를 얼마나 벗어나는지 측정하였다.

분석한 Feature는 다음과 같다.

- 전체 공정 변수의 평균 절대 Z-score
- 최대 절대 Z-score
- 절대 Z-score의 표준편차
- |Z| ≥ 1.5인 공정 변수 개수
- |Z| ≥ 2.0인 공정 변수 개수
- |Z| ≥ 2.5인 공정 변수 개수

그러나 모든 Feature의 PR-AUC가 약 0.035~0.041 수준으로 나타났다.

전체 RG3 불량률이 약 0.0423이므로,
해당 Feature들은 Random Ranking과 유사하거나
그보다 낮은 수준의 분리력을 보였다.

따라서 RG3 불량은 여러 센서가 동시에
정상 범위를 크게 벗어나는 단순한 이상치 형태로
설명하기 어려운 것으로 판단하였다.

이후에는 전체적인 이상치 크기보다
개별 공정 변수의 특정 방향 또는 분포 형태가
불량과 관련되는지 분석한다.

In [32]:
ks_top15 = (
    feature_signal[
        [
            "Feature",
            "Normal_Mean",
            "Defect_Mean",
            "Abs_Mean_Diff",
            "KS_Stat",
            "KS_pvalue"
        ]
    ]
    .sort_values(
        "KS_Stat",
        ascending=False
    )
    .head(15)
)

ks_top15

,Feature,Normal_Mean,Defect_Mean,Abs_Mean_Diff,KS_Stat,KS_pvalue
18,Barrel_Temperature_5,-0.019687,0.445713,0.465400,0.188975,0.317212
22,Mold_Temperature_4,0.003555,-0.080492,0.084048,0.168905,0.450972
20,Hopper_Temperature,0.005919,-0.134011,0.139930,0.150954,0.591757
21,Mold_Temperature_3,0.003902,-0.088344,0.092246,0.145936,0.633356
7,Max_Injection_Speed,0.001844,-0.041750,0.043595,0.134205,0.730619
2,Plasticizing_Time,-0.007639,0.172956,0.180595,0.124099,0.809222
13,Average_Back_Pressure,-0.002660,0.060221,0.062881,0.121979,0.825106
16,Barrel_Temperature_3,0.006064,-0.137298,0.143362,0.119011,0.845966
19,Barrel_Temperature_6,-0.001925,0.043579,0.045504,0.112721,0.886518
17,Barrel_Temperature_4,0.002025,-0.045846,0.047871,0.106360,0.921549


In [33]:
rg3_rh = rg3_rh.copy()

print("Shape:", rg3_rh.shape)
print("Defect Rate:", rg3_rh["PassOrFail"].mean())

Shape: (591, 158)
Defect Rate: 0.04230118443316413


In [34]:
absz_top15 = (
    feature_signal[
        [
            "Feature",
            "PR_Abs_Normal_Z",
            "Best_Direction_PR",
            "KS_Stat"
        ]
    ]
    .sort_values(
        "PR_Abs_Normal_Z",
        ascending=False
    )
    .head(15)
)

absz_top15

,Feature,PR_Abs_Normal_Z,Best_Direction_PR,KS_Stat
17,Barrel_Temperature_4,0.057495,0.054750,0.106360
12,Max_Back_Pressure,0.053223,0.073926,0.074700
14,Barrel_Temperature_1,0.052191,0.054049,0.084028
10,Max_Injection_Pressure,0.050608,0.046152,0.067845
7,Max_Injection_Speed,0.049449,0.048256,0.134205
13,Average_Back_Pressure,0.047151,0.053974,0.121979
18,Barrel_Temperature_5,0.046473,0.064997,0.188975
22,Mold_Temperature_4,0.043437,0.079263,0.168905
8,Max_Screw_RPM,0.043111,0.045765,0.067067
20,Hopper_Temperature,0.042729,0.050017,0.150954


## 25. 공정 변수 Pair Interaction 탐색

개별 Original Feature의 정상/불량 분리력을 분석한 결과,
단일 변수만으로 강한 불량 신호를 나타내는 Feature는 확인되지 않았다.

따라서 RG3 불량은 하나의 공정 변수보다
두 개 이상의 공정 조건이 동시에 특정 관계를 형성할 때
발생할 가능성을 검토한다.

23개의 Original Process Feature에서 가능한 모든 2개 변수 조합에 대해
다음 Interaction을 생성한다.

- 두 변수의 합
- 두 변수의 차이
- 두 변수 차이의 절댓값
- 두 변수의 곱

단, 전체 데이터의 Target을 이용하여 Feature를 먼저 선택하면
Validation 정보가 유입될 수 있으므로,
각 Cross Validation Fold의 Train 데이터에서만
Interaction Feature의 분리력을 계산한다.

Train에서 높은 분리력을 보인 상위 Interaction Feature만 선택한 뒤,
Original Feature와 함께 XGBoost에 입력하여 Validation 성능을 평가한다.

In [35]:
from itertools import combinations
# ==========================================
# Pair Interaction Feature 생성
# ==========================================

pair_feature_dict = {}
pair_feature_meta = []

for col_a, col_b in combinations(
    PROCESS_COLS,
    2
):

    a = rg3_rh[col_a]
    b = rg3_rh[col_b]

    feature_defs = {

        f"{col_a}__PLUS__{col_b}":
            a + b,

        f"{col_a}__MINUS__{col_b}":
            a - b,

        f"{col_a}__ABS_DIFF__{col_b}":
            (a - b).abs(),

        f"{col_a}__MULTIPLY__{col_b}":
            a * b
    }

    for feature_name, values in feature_defs.items():

        pair_feature_dict[
            feature_name
        ] = values

        pair_feature_meta.append({
            "Feature": feature_name,
            "Feature_A": col_a,
            "Feature_B": col_b,
            "Transform":
                feature_name
                .split("__")[1]
        })

In [36]:
pair_features = pd.DataFrame(
    pair_feature_dict,
    index=rg3_rh.index
)

pair_feature_meta = pd.DataFrame(
    pair_feature_meta
)

print(
    "Pair Feature Shape:",
    pair_features.shape
)

print(
    "Pair Feature Count:",
    pair_features.shape[1]
)

print(
    "Missing:",
    pair_features
    .isna()
    .sum()
    .sum()
)

Pair Feature Shape: (591, 1012)
Pair Feature Count: 1012
Missing: 0


In [37]:
y_pair = (
    rg3_rh["PassOrFail"]
    .reset_index(drop=True)
)

X_original_pair = (
    rg3_rh[PROCESS_COLS]
    .reset_index(drop=True)
)

X_pair = (
    pair_features
    .reset_index(drop=True)
)

In [38]:
pair_cv_results = []

pair_selection_count = {}

TOP_K_LIST = [
    3,
    5,
    10
]

In [39]:
for fold_num, (train_idx, valid_idx) in enumerate(
    cv_splits,
    start=1
):

    y_train = y_pair.iloc[
        train_idx
    ]

    y_valid = y_pair.iloc[
        valid_idx
    ]

    # ======================================
    # 1. Train 데이터에서 Pair Feature Ranking
    # ======================================

    train_pair_scores = []

    for col in X_pair.columns:

        train_values = (
            X_pair[col]
            .iloc[train_idx]
            .values
        )

        # 값이 클수록 불량
        pr_high = average_precision_score(
            y_train,
            train_values
        )

        # 값이 작을수록 불량
        pr_low = average_precision_score(
            y_train,
            -train_values
        )

        best_pr = max(
            pr_high,
            pr_low
        )

        train_pair_scores.append(
            (
                col,
                best_pr
            )
        )

    train_pair_scores = sorted(
        train_pair_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # ======================================
    # 2. Original Baseline
    # ======================================

    X_train_base = (
        X_original_pair
        .iloc[train_idx]
    )

    X_valid_base = (
        X_original_pair
        .iloc[valid_idx]
    )

    neg = (
        y_train == 0
    ).sum()

    pos = (
        y_train == 1
    ).sum()

    pos_weight = (
        neg / pos
    )

    base_model = XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=pos_weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    base_model.fit(
        X_train_base,
        y_train
    )

    base_score = (
        base_model
        .predict_proba(
            X_valid_base
        )[:, 1]
    )

    pair_cv_results.append({
        "Fold": fold_num,
        "Feature_Set":
            "Original",

        "PR_AUC":
            average_precision_score(
                y_valid,
                base_score
            ),

        "ROC_AUC":
            roc_auc_score(
                y_valid,
                base_score
            )
    })

    # ======================================
    # 3. Top-K Pair Feature 추가
    # ======================================

    for top_k in TOP_K_LIST:

        selected_pair_cols = [
            x[0]
            for x in train_pair_scores[
                :top_k
            ]
        ]

        # 선택 빈도 저장
        for col in selected_pair_cols:

            pair_selection_count[col] = (
                pair_selection_count
                .get(col, 0)
                + 1
            )

        # Original
        X_train_original = (
            X_original_pair
            .iloc[train_idx]
            .reset_index(drop=True)
        )

        X_valid_original = (
            X_original_pair
            .iloc[valid_idx]
            .reset_index(drop=True)
        )

        # Pair
        X_train_selected = (
            X_pair[
                selected_pair_cols
            ]
            .iloc[train_idx]
            .reset_index(drop=True)
        )

        X_valid_selected = (
            X_pair[
                selected_pair_cols
            ]
            .iloc[valid_idx]
            .reset_index(drop=True)
        )

        # 결합
        X_train_combined = pd.concat(
            [
                X_train_original,
                X_train_selected
            ],
            axis=1
        )

        X_valid_combined = pd.concat(
            [
                X_valid_original,
                X_valid_selected
            ],
            axis=1
        )

        model = XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train_combined,
            y_train.reset_index(
                drop=True
            )
        )

        valid_score = (
            model
            .predict_proba(
                X_valid_combined
            )[:, 1]
        )

        pair_cv_results.append({
            "Fold": fold_num,

            "Feature_Set":
                f"Original + Pair Top{top_k}",

            "PR_AUC":
                average_precision_score(
                    y_valid,
                    valid_score
                ),

            "ROC_AUC":
                roc_auc_score(
                    y_valid,
                    valid_score
                )
        })

In [40]:
pair_cv_results = pd.DataFrame(
    pair_cv_results
)

pair_cv_summary = (
    pair_cv_results
    .groupby(
        "Feature_Set"
    )
    .agg(
        PR_AUC_Mean=(
            "PR_AUC",
            "mean"
        ),

        PR_AUC_Std=(
            "PR_AUC",
            "std"
        ),

        PR_AUC_Min=(
            "PR_AUC",
            "min"
        ),

        PR_AUC_Max=(
            "PR_AUC",
            "max"
        ),

        ROC_AUC_Mean=(
            "ROC_AUC",
            "mean"
        ),

        ROC_AUC_Std=(
            "ROC_AUC",
            "std"
        )
    )
    .sort_values(
        "PR_AUC_Mean",
        ascending=False
    )
)

pair_cv_summary

,PR_AUC_Mean,PR_AUC_Std,PR_AUC_Min,PR_AUC_Max,ROC_AUC_Mean,ROC_AUC_Std
Feature_Set,,,,,,
Original,0.080049,0.049817,0.031940,0.234006,0.521028,0.073881
Original + Pair Top3,0.079291,0.052975,0.034487,0.285827,0.518913,0.076988
Original + Pair Top10,0.078274,0.049922,0.034418,0.208871,0.499872,0.083459
Original + Pair Top5,0.076040,0.039150,0.035371,0.222951,0.521607,0.086307


In [41]:
base_pair_scores = (
    pair_cv_results[
        pair_cv_results[
            "Feature_Set"
        ] == "Original"
    ][
        [
            "Fold",
            "PR_AUC"
        ]
    ]
    .rename(
        columns={
            "PR_AUC":
                "Original_PR_AUC"
        }
    )
)

In [42]:
pair_win_rows = []

for feature_set in [
    "Original + Pair Top3",
    "Original + Pair Top5",
    "Original + Pair Top10"
]:

    temp = (
        pair_cv_results[
            pair_cv_results[
                "Feature_Set"
            ] == feature_set
        ][
            [
                "Fold",
                "PR_AUC"
            ]
        ]
        .rename(
            columns={
                "PR_AUC":
                    "Pair_PR_AUC"
            }
        )
    )

    compare = (
        base_pair_scores
        .merge(
            temp,
            on="Fold"
        )
    )

    compare["Delta"] = (
        compare["Pair_PR_AUC"]
        - compare[
            "Original_PR_AUC"
        ]
    )

    pair_win_rows.append({
        "Feature_Set":
            feature_set,

        "Win":
            int(
                (
                    compare["Delta"]
                    > 0
                ).sum()
            ),

        "Tie":
            int(
                (
                    compare["Delta"]
                    == 0
                ).sum()
            ),

        "Lose":
            int(
                (
                    compare["Delta"]
                    < 0
                ).sum()
            ),

        "Mean_Delta":
            compare[
                "Delta"
            ].mean(),

        "Median_Delta":
            compare[
                "Delta"
            ].median()
    })

In [43]:
pair_win_summary = (
    pd.DataFrame(
        pair_win_rows
    )
    .sort_values(
        "Mean_Delta",
        ascending=False
    )
)

pair_win_summary

,Feature_Set,Win,Tie,Lose,Mean_Delta,Median_Delta
0,Original + Pair Top3,14,0,16,-0.000758,-0.000570
2,Original + Pair Top10,12,0,18,-0.001775,-0.003799
1,Original + Pair Top5,16,0,14,-0.004010,0.000270


In [44]:
pair_selection_summary = (
    pd.DataFrame(
        pair_selection_count.items(),
        columns=[
            "Feature",
            "Selected_Count"
        ]
    )
    .sort_values(
        "Selected_Count",
        ascending=False
    )
)

In [45]:
pair_selection_summary.head(20)

,Feature,Selected_Count
11,Plasticizing_Time__MULTIPLY__Average_Screw_RPM,47
0,Max_Injection_Pressure__MULTIPLY__Barrel_Tempe...,44
2,Max_Injection_Speed__MULTIPLY__Mold_Temperature_4,40
12,Plasticizing_Time__ABS_DIFF__Average_Screw_RPM,28
20,Cushion_Position__PLUS__Mold_Temperature_3,27
22,Cushion_Position__PLUS__Mold_Temperature_4,22
21,Max_Injection_Speed__MULTIPLY__Mold_Temperature_3,18
15,Max_Screw_RPM__PLUS__Barrel_Temperature_5,16
10,Max_Injection_Pressure__ABS_DIFF__Barrel_Tempe...,16
13,Barrel_Temperature_5__PLUS__Mold_Temperature_3,15


In [46]:
pair_selection_detail = (
    pair_selection_summary
    .merge(
        pair_feature_meta,
        on="Feature",
        how="left"
    )
)

pair_selection_detail.head(20)

,Feature,Selected_Count,Feature_A,Feature_B,Transform
0,Plasticizing_Time__MULTIPLY__Average_Screw_RPM,47,Plasticizing_Time,Average_Screw_RPM,MULTIPLY
1,Max_Injection_Pressure__MULTIPLY__Barrel_Tempe...,44,Max_Injection_Pressure,Barrel_Temperature_4,MULTIPLY
2,Max_Injection_Speed__MULTIPLY__Mold_Temperature_4,40,Max_Injection_Speed,Mold_Temperature_4,MULTIPLY
3,Plasticizing_Time__ABS_DIFF__Average_Screw_RPM,28,Plasticizing_Time,Average_Screw_RPM,ABS_DIFF
4,Cushion_Position__PLUS__Mold_Temperature_3,27,Cushion_Position,Mold_Temperature_3,PLUS
5,Cushion_Position__PLUS__Mold_Temperature_4,22,Cushion_Position,Mold_Temperature_4,PLUS
6,Max_Injection_Speed__MULTIPLY__Mold_Temperature_3,18,Max_Injection_Speed,Mold_Temperature_3,MULTIPLY
7,Max_Screw_RPM__PLUS__Barrel_Temperature_5,16,Max_Screw_RPM,Barrel_Temperature_5,PLUS
8,Max_Injection_Pressure__ABS_DIFF__Barrel_Tempe...,16,Max_Injection_Pressure,Barrel_Temperature_4,ABS_DIFF
9,Barrel_Temperature_5__PLUS__Mold_Temperature_3,15,Barrel_Temperature_5,Mold_Temperature_3,PLUS
